# Семинар 3. Кластеризация: алгоритмы, параметры и проверка результата

Основной маршрут: k-means → DBSCAN и k-distance →
агломеративная кластеризация → Connectivity Constraint → метрики → Квантование цветов.

Нужны NumPy, pandas, SciPy, Matplotlib, scikit-learn. При необходимости отдельно
выполните `%pip install numpy pandas scipy matplotlib scikit-learn`, затем перезапустите ядро.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import (
    KMeans,
    MiniBatchKMeans,
    DBSCAN,
    AgglomerativeClustering,
)
from sklearn.datasets import (
    make_blobs,
    make_moons,
    make_swiss_roll,
    load_digits,
    load_sample_image,
)
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.metrics import (
    rand_score,
    silhouette_score,
    davies_bouldin_score,
    homogeneity_score,
    mutual_info_score,
)
from scipy.sparse.csgraph import connected_components
from sklearn.metrics.cluster import contingency_matrix

SEED = 42
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": False})


def show_partition(ax, X, labels, title):
    labels = np.asarray(labels)
    valid = labels != -1
    max_label = labels[valid].max() if np.any(valid) else 0
    ax.scatter(
        X[valid, 0],
        X[valid, 1],
        c=labels[valid],
        cmap="tab10" if max_label < 10 else "gist_rainbow",
        s=22,
        vmin=0,
        vmax=max(9, max_label),
    )
    if np.any(~valid):
        ax.scatter(
            X[~valid, 0], X[~valid, 1], c="gray", marker="x", s=25, label="Шум"
        )
        ax.legend(fontsize=8)
    ax.set(title=title, xlabel="x₁", ylabel="x₂")
    ax.set_aspect("equal", adjustable="datalim")

## 1. k-means
Целевая функция:
$$J=\sum_{i=1}^{N}\|x_i-\mu_{z_i}\|^2.$$
Чередуем назначение точек ближайшим центрам и пересчёт средних (алгоритм Ллойда).
Каждый шаг не увеличивает J. При согласованных правилах назначения и обработки
пустых кластеров достигаем устойчивого решения, но не гарантируем глобальный минимум.
Рестарты нужны из-за зависимости от начальных центров.

**Соглашения:**
- При равных расстояниях выбираем центр с меньшим индексом.
- Если кластер пуст, сохраняем предыдущий центр; не считаем среднее пустого массива.
- Останавливаемся, когда назначения перестали меняться; дополнительно задаём `max_iter`.

### Инициализация и рестарты
Используем одно и то же облако и k. Сравним несколько одиночных случайных стартов
с k-means++ и несколькими рестартами. 

In [ ]:
blob_X, blob_y = make_blobs(
    n_samples=300, centers=4, cluster_std=0.8, random_state=SEED
)
starts = []
for seed in range(8):
    model = KMeans(
        n_clusters=4, init="random", n_init=1, random_state=seed
    ).fit(blob_X)
    starts.append((model.inertia_, model.labels_))
best_start = min(starts, key=lambda result: result[0])
worst_start = max(starts, key=lambda result: result[0])
plus_model = KMeans(
    n_clusters=4, init="k-means++", n_init=10, random_state=SEED
).fit(blob_X)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (J, labels), name in zip(
    axes,
    [worst_start, best_start, (plus_model.inertia_, plus_model.labels_)],
    ["Худший random-start", "Лучший random-start", "k-means++, 10 стартов"],
):
    show_partition(ax, blob_X, labels, f"{name}\nJ={J:.1f}")
plt.tight_layout()
plt.show()

## 2. DBSCAN: core, border и noise
Фиксируем $N_\varepsilon(x)=\{y:d(x,y)\leq\varepsilon\}$; **сама точка входит** в окрестность.
*Core*: $|N_\varepsilon(x)|\geq m$, где `m = min_samples`.
*Border*: не core, но находится в ε-окрестности core-точки. Остальные — *noise*.

- Border рядом с двумя core-компонентами присоединяется к той, что обработана первой.
  Такая точка не соединяет core-компоненты в один кластер (только core-точки могут продолжать цепочку, которая объединяет кластер).

### Как подбирать ε: k-distance
Сначала фиксируем `m=5`. Для каждой точки находим расстояние до **m-й точки,
считая её саму**, сортируем эти расстояния и ищем изгиб.
Можно сделать через knn.

Вызываем `kneighbors(X)` явно: запросы включают обучающие точки и их собственное
нулевое расстояние.

In [ ]:
moon_X, moon_y = make_moons(n_samples=300, noise=0.04, random_state=SEED)
min_samples = 5
neighbors = NearestNeighbors(n_neighbors=min_samples).fit(moon_X)
neighbor_distances, _ = neighbors.kneighbors(moon_X)
k_distances = np.sort(neighbor_distances[:, -1])
# Визуальная гипотеза о положении изгиба, не автоматическая оценка.
suggested_eps = 0.11
eps_values = [0.08, suggested_eps, 0.50]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].plot(k_distances)
axes[0].set(
    title="k-distance, m=5 (включая себя)",
    xlabel="Точки по возрастанию расстояния",
    ylabel="Расстояние",
)
for eps, ax in zip(eps_values, axes[1:]):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(moon_X)
    if eps <= k_distances.max() * 1.2:
        axes[0].axhline(eps, ls="--", label=f"ε={eps}")
    count = len(set(labels) - {-1})
    show_partition(
        ax,
        moon_X,
        labels,
        f"ε={eps}: k={count}, noise={np.mean(labels == -1):.0%}",
    )
elbow_index = int(np.argmin(np.abs(k_distances - suggested_eps)))
axes[0].scatter(elbow_index, k_distances[elbow_index], c="red", zorder=5)
axes[0].annotate(
    "Предполагаемый изгиб: ε ≈ 0.11",
    xy=(elbow_index, k_distances[elbow_index]),
    xytext=(0.05, 0.75),
    textcoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "red"},
    fontsize=9,
)
axes[0].set_ylim(0, k_distances.max() * 1.2)
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Агломеративная кластеризация: обновляем расстояния
Начинаем с одноточечных кластеров; на каждом шаге объединяем ближайшую пару.
- При равенстве linkage задается детерминированный порядок пар.
- Расстояния до нового кластера вычисляем по расстояниям, которые были до слияния.
- Также обновляем размеры кластеров и маску активных кластеров.

### Два слияния вручную
Объекты на прямой: 0, 2, 5, 9. Проверьте через библиотеку. Столбцы Z: ID двух слитых кластеров,
высота слияния, число объектов. Новым кластерам присваиваются ID n, n+1, …

На дендрограммах пунктиром отмечен общий учебный порог **3.5**.
Цветные ветви ниже порога показывают отдельные кластеры; слияния выше порога — серые.
Число кластеров равно числу ветвей, пересечённых горизонтальной линией.
Одинаковый порог у разных linkage не обязан давать одинаковое число кластеров.

In [ ]:
line_X = np.array([[0.0], [2.0], [5.0], [9.0]])
print("Матрица расстояний:\n", squareform(pdist(line_X)))
# Один учебный порог для сравнения трёх linkage.
cut_height = 3.5
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
for method, ax in zip(["single", "complete", "average"], axes):
    Z = linkage(line_X, method=method)
    print(method, "\n", Z)
    dendrogram(
        Z,
        labels=["0", "2", "5", "9"],
        color_threshold=cut_height,
        above_threshold_color="gray",
        ax=ax,
    )
    ax.axhline(
        cut_height,
        color="black",
        ls="--",
        label=f"Порог = {cut_height}",
    )
    # Порог не совпадает ни с одной высотой слияния.
    cluster_count = len(line_X) - np.count_nonzero(Z[:, 2] < cut_height)
    ax.set(
        title=f"{method}: {cluster_count} кластера",
        xlabel="Исходные точки",
        ylabel="Высота linkage",
    )
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Connectivity constraint
В обычной агломеративной кластеризации разрешены слияния любых кластеров. В качестве дополнительного условия можно ввести ограничение связности (**сonnectivity constraint**): объединять кластеры только тогда, когда минимальное расстояние между ними меньше заранее заданной константы. Можно представить в виде графа соседства: задаём порог $r$ и соединяем объекты ребром, если $d(x,y) < r$.
Тогда условие допустимости слияния кластеров A и B:
$$\min_{x\in A,\;y\in B}d(x,y)<r.$$
Это ограничение по ближайшей паре точек, а не порог на значение linkage.

**В примере ниже** используется другой способ построения графа:
**kNN-граф**: *граф, в котором вершины — объекты, а рёбра соединяют соседние объекты*.

Важно: в случае несвязного графа AgglomerativeClustering в sklearn выдаёт предупреждение и добавляет связи между компонентами, чтобы дерево слияний могло продолжаться. Если же нужно строго запретить объединение разных компонент, их следует кластеризовать отдельно.

In [ ]:
roll_X, roll_t = make_swiss_roll(n_samples=600, noise=0.03, random_state=SEED)
roll_X = roll_X / 10.0
connectivity = kneighbors_graph(roll_X, n_neighbors=12, include_self=False)
connectivity = connectivity.maximum(connectivity.T)
components = connected_components(
    connectivity, directed=False, return_labels=False
)
print("Компонент kNN-графа:", components)
assert components == 1, "Для этой демонстрации нужен связный граф"
fig = plt.figure(figsize=(14, 4))
for column, (title, graph) in enumerate(
    [
        ("Исходная геометрия (цвет - положение вдоль рулона)", None),
        ("Ward без connectivity", None),
        ("Ward с kNN-connectivity", connectivity),
    ]
):
    ax = fig.add_subplot(1, 3, column + 1, projection="3d")
    colors = (
        roll_t
        if column == 0
        else AgglomerativeClustering(
            n_clusters=6, linkage="ward", connectivity=graph
        ).fit_predict(roll_X)
    )
    ax.scatter(
        roll_X[:, 0],
        roll_X[:, 1],
        roll_X[:, 2],
        c=colors,
        cmap="tab10" if column else "viridis",
        s=7,
    )
    ax.view_init(elev=12, azim=-65)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 5. Метрики: что именно считается хорошим кластером?
### Внутренние: без истинных меток
Силуэт:
$$s(x)=\frac{b(x)-a(x)}{\max(a(x),b(x))}.$$
$a(x)$ — среднее расстояние до остальных в своём кластере;
$b(x)$ — минимальное среднее расстояние до другого кластера. Чем больше средний силуэт, тем лучше по этому критерию.

**Davies–Bouldin** (меньше — лучше):
$$S_i=\frac1{|C_i|}\sum_{x\in C_i}\|x-\mu_i\|,\qquad
DB=\frac1k\sum_i\max_{j\ne i}\frac{S_i+S_j}{\|\mu_i-\mu_j\|}.$$
Это средняя «худшая похожесть» кластера на другой: большой разброс и близкие центры
ухудшают результат.

**Dunn** (больше — лучше):
$$Dunn=\frac{\min_{i\ne j}\min_{x\in C_i,y\in C_j}d(x,y)}
{\max_i\max_{x,y\in C_i}d(x,y)}.$$
Числитель — ближайшая межкластерная пара; знаменатель — максимальный диаметр.
Одна близкая межкластерная пара или далёкий выброс может резко снизить индекс.


Шум DBSCAN не считаем обычным кластером во внутренних метриках.
Ниже выводим coverage; если шум исключён, метрики условны на оставшемся подмножестве.
Для строгого сравнения алгоритмов используйте одинаковые объекты.
В основном примере $\varepsilon=0.2$, шумовых точек нет.

In [ ]:
def dunn_index(X, labels):
    labels = np.asarray(labels)
    clusters = np.unique(labels)
    if not 2 <= len(clusters) < len(labels):
        return np.nan
    distances = squareform(pdist(X))
    masks = [labels == cluster for cluster in clusters]
    diameter = max(distances[np.ix_(mask, mask)].max() for mask in masks)
    if diameter == 0:
        return np.nan
    gap = min(
        distances[np.ix_(masks[i], masks[j])].min()
        for i in range(len(masks))
        for j in range(i + 1, len(masks))
    )
    return gap / diameter


def internal_scores(X, labels):
    labels = np.asarray(labels)
    keep = labels != -1
    kept_X, kept_labels = X[keep], labels[keep]
    count = len(np.unique(kept_labels))
    result = {
        "clusters": count,
        "coverage": np.mean(keep),
        "silhouette ↑": np.nan,
        "DB ↓": np.nan,
        "Dunn ↑": np.nan,
    }
    if 2 <= count < len(kept_labels):
        result["silhouette ↑"] = silhouette_score(kept_X, kept_labels)
        result["DB ↓"] = davies_bouldin_score(kept_X, kept_labels)
        result["Dunn ↑"] = dunn_index(kept_X, kept_labels)
    return result


# Sanity checks: silhouette/Dunn для одного кластера не вычисляем.
assert np.isnan(dunn_index(line_X, np.zeros(len(line_X), dtype=int)))
assert (
    internal_scores(line_X, -np.ones(len(line_X), dtype=int))["coverage"] == 0
)

### Полумесяцы: согласуются ли метрики?
Rand index использует истинные метки; остальные индексы их не видят.

In [ ]:
moon_partitions = {
    "k-means": KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(
        moon_X
    ),
    "DBSCAN": DBSCAN(eps=0.2, min_samples=5).fit_predict(moon_X),
    "Average linkage": AgglomerativeClustering(
        n_clusters=2, linkage="average"
    ).fit_predict(moon_X),
}
metric_rows = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for (name, labels), ax in zip(moon_partitions.items(), axes):
    metric_rows.append(
        {
            "method": name,
            **internal_scores(moon_X, labels),
            "Rand ↑": rand_score(moon_y, labels),
        }
    )
    show_partition(ax, moon_X, labels, name)
plt.tight_layout()
plt.show()
print(pd.DataFrame(metric_rows).round(3).to_string(index=False))
assert np.all(moon_partitions["DBSCAN"] != -1)

### Внешние: Purity, Homogeneity, Rand index и Mutual Information
Пусть $Y$ — истинный класс, $C$ — кластер, $n_{yc}$ — число объектов класса $y$ в кластере $c$.
Все четыре метрики требуют истинных меток и не меняются при перестановке ID кластеров.

**Purity** — доля объектов, согласных с наиболее частым классом своего кластера:
$$\mathrm{Purity}=\frac1N\sum_c\max_y n_{yc}.$$
Больше — лучше по этому критерию. Singleton-разбиение даёт purity=1,
поэтому один этот показатель не подходит для выбора числа кластеров.

**Homogeneity** — насколько кластеры чисты по истинным классам:
$$h=1-\frac{H(Y\mid C)}{H(Y)}.$$
Здесь $H(Y)=-\sum_y p_y\ln p_y$ — неопределённость класса до знания кластера,
а $H(Y\mid C)=\sum_c p_c H(Y\mid C=c)$ — оставшаяся неопределённость.
Если каждый кластер содержит только один класс, условная энтропия равна 0 и h=1.
Диапазон h — от 0 до 1.
Purity и homogeneity оценивают чистоту разными способами; это не синонимы.

**Rand index** сравнивает все неупорядоченные пары различных объектов:
$$\mathrm{RI}=\frac{a+b}{\binom{N}{2}}.$$
Здесь a — число пар, находящихся вместе и по классам, и по кластерам;
b — число пар, находящихся раздельно в обоих разбиениях.
Это доля согласованных решений о парах: от 0 до 1, больше — лучше.
Случайное разбиение не обязано давать RI около 0: поправки на случайность здесь нет.
Большое число пар из разных групп может давать высокий RI даже слабому разбиению (например, если каждая точка - свой кластер, b большой $\rightarrow$ Rand высокий).

**Mutual information (MI)** — сколько информации о классе даёт знание кластера:
$$I(Y;C)=\sum_{y,c:p_{yc}>0}p_{yc}\ln\frac{p_{yc}}{p_y p_c}
=H(Y)-H(Y\mid C),$$
где $p_{yc}=\frac{n_{yc}}{N}$, $p_y=\frac{n_y}{N}$, $p_c=\frac{n_c}{N}$.
При независимости класса и кластера $MI=0$. MI не нормирована к [0,1] и не скорректирована на случайность.
$I(Y;C)\leq H(Y)$;
singleton-разбиение достигает $H(Y)$, поэтому высокая MI не гарантирует полезного числа кластеров.
При $H(Y)>0$ homogeneity равна $\frac{MI}{H(Y)}$.

Перед запуском сравните один кластер, singleton-разбиение и раздробленные чистые классы.
Какие метрики поощряют дробление? Почему нельзя выбирать число кластеров только по чистоте?

In [ ]:
def purity_score(truth, labels):
    table = contingency_matrix(truth, labels)
    return table.max(axis=0).sum() / table.sum()


def external_scores(truth, labels):
    return {
        "purity": purity_score(truth, labels),
        "homogeneity": homogeneity_score(truth, labels),
        "Rand": rand_score(truth, labels),
        "MI": mutual_info_score(truth, labels),
    }


true_groups = np.array([0, 0, 0, 1, 1, 1])
examples = {
    "Идеально": true_groups,
    "Переставлены номера": 1 - true_groups,
    "Один кластер": np.zeros(6, dtype=int),
    "Каждый отдельно": np.arange(6),
    "Чистые, но раздробленные": np.array([0, 0, 1, 2, 2, 3]),
}
external_table = pd.DataFrame(
    [
        {"partition": name, **external_scores(true_groups, labels)}
        for name, labels in examples.items()
    ]
)
print(external_table.round(3).to_string(index=False))
assert external_scores(true_groups, true_groups) == external_scores(
    true_groups, 1 - true_groups
)
assert external_scores(true_groups, true_groups)["Rand"] == 1.0
assert np.isclose(external_scores(true_groups, true_groups)["MI"], np.log(2))
assert external_scores(true_groups, np.zeros(6, dtype=int))["MI"] == 0.0

### Digits: центры кластеров и внешняя оценка
Кластеризуем изображения без использования цифр-меток. Затем смотрим на центры
и оцениваем согласие разбиения с метками через purity, homogeneity, Rand index и mutual information.

Это **не классификатор, оценённый на новых объектах**: кластеризация и оценка здесь
на одном наборе. 

In [ ]:
digits = load_digits()
digit_model = KMeans(n_clusters=10, n_init=10, random_state=SEED)
digit_labels = digit_model.fit_predict(digits.data)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for cluster, ax in enumerate(axes.ravel()):
    ax.imshow(
        digit_model.cluster_centers_[cluster].reshape(8, 8), cmap="gray_r"
    )
    ax.set_title(f"Кластер {cluster}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Digits: лучше ли результат в сравнении со случайным разбиением?
Отдельное число без baseline трудно интерпретировать. Сравним k-means с 30 случайными
разбиениями: перемешиваем найденные cluster labels между объектами. При этом **число
кластеров и размер каждого кластера сохраняются**, а связь с изображениями разрушается.
Истинные метки цифр остаются на месте; ни один baseline не используется для обучения.

Покажем среднее и стандартное отклонение случайных результатов. Это разброс
случайных разбиений при фиксированных данных и размерах кластеров.

«Идеальное совпадение» — результат при clusters = true labels. В частности, максимальная MI для этих меток равна H(Y).

До запуска: должен ли Rand случайного разбиения быть близок к нулю?
После запуска: какие результаты k-means действительно отличаются от baseline?

In [ ]:
baseline_rng = np.random.default_rng(SEED)
random_runs = 30
random_digit_rows = []
for repeat in range(random_runs):
    random_labels = baseline_rng.permutation(digit_labels)
    assert np.array_equal(
        np.bincount(random_labels), np.bincount(digit_labels)
    )
    random_digit_rows.append(external_scores(digits.target, random_labels))

random_digit_scores = pd.DataFrame(random_digit_rows)
kmeans_digit_scores = pd.Series(external_scores(digits.target, digit_labels))
ideal_digit_scores = pd.Series(external_scores(digits.target, digits.target))
random_mean = random_digit_scores.mean()
random_std = random_digit_scores.std(ddof=1)
digit_comparison = pd.DataFrame(
    [kmeans_digit_scores, random_mean, random_std, ideal_digit_scores],
    index=[
        "k-means",
        "Случайное: среднее (30 разбиений)",
        "Случайное: стандартное отклонение",
        "Идеальное совпадение (ориентир)",
    ],
)
print(digit_comparison.round(4).to_string())

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for metric, ax in zip(kmeans_digit_scores.index, axes):
    ax.bar(
        [0, 1],
        [random_mean[metric], kmeans_digit_scores[metric]],
        color=["gray", "tab:blue"],
        alpha=0.8,
    )
    ax.errorbar(
        0,
        random_mean[metric],
        yerr=random_std[metric],
        fmt="none",
        color="black",
        capsize=5,
    )
    ax.axhline(
        ideal_digit_scores[metric],
        color="black",
        ls="--",
        label="Идеальное совпадение",
    )
    ax.set(
        title=metric,
        xticks=[0, 1],
        xticklabels=["Случайное", "k-means"],
        ylim=(0, ideal_digit_scores[metric] * 1.12),
    )
    ax.legend(fontsize=7, loc="upper right")
fig.suptitle("Digits: случайное разбиение с теми же размерами кластеров")
plt.tight_layout()
plt.show()

**Как читать сравнение:**
- Purity и homogeneity; MI: стали ли кластеры заметно чище случайных?
- Rand: даже случайный результат может быть высоким, поскольку большинство пар
  относятся к разным цифрам и разным кластерам. Сравниваем с baseline, а не с нулём.

## 6. Цветовая кластеризация изображения
Каждый пиксель — точка в RGB. **Координаты пикселя не используются**:
два одинаковых цвета в разных частях изображения — одинаковые объекты для модели.
Центры кластеров образуют палитру, каждый пиксель заменяем ближайшим цветом палитры.

### MiniBatchKMeans
Допустим, мы хотим заменить цвета изображения палитрой из **16 цветов**.
Каждый центр кластера — один цвет этой палитры.

Обычный **KMeans** на каждой итерации просматривает все обучающие пиксели:
для каждого выбирает ближайший цвет палитры, затем пересчитывает каждый цвет
как среднее RGB назначенных ему пикселей. Если пикселей много, такие проходы занимают время.

**MiniBatchKMeans** работает небольшими порциями:
1. Случайно выбирает порцию обучающих пикселей (batch) — например, 256.
2. Для каждого пикселя из этой порции находит ближайший центроид.
3. Обновляет центроиды с учётом новых пикселей.
4. Берёт следующую случайную порцию и продолжает с уже обновлённой палитрой.

Обработка маленьких порций часто ускоряет обучение на больших данных.
Цена ускорения — приближённые обновления: ошибка на всех пикселях может временно
расти, а итоговая палитра может давать большую ошибку, чем обычный KMeans.

После обучения оба метода восстанавливают изображение одинаково:
заменяют каждый пиксель ближайшим цветом найденной палитры.

[Подробнее: MiniBatchKMeans в sklearn](https://scikit-learn.org/stable/modules/clustering.html#mini-batch-kmeans).

### Что сравниваем
Используем **полное изображение 427 × 640: 273280 пикселей**. Оба метода
обучаются на всех пикселях; RGB делим на 255. Сравним палитры из 8, 16 и 64 цветов.
Cчитаем **MSE восстановления** - средний квадрат разницы между исходными цветами пикселей и цветами после замены палитрой.

У MiniBatchKMeans `batch_size=1024`: случайные порции выбираются из **всего** изображения.
Полная MSE при обучении mini-batch не обязана
монотонно уменьшаться.

В итоговой таблице `fit_mean_s` — среднее время пяти обучений после прогрева.


In [ ]:
from time import perf_counter
from sklearn.cluster import kmeans_plusplus
from threadpoolctl import threadpool_limits

image = load_sample_image("china.jpg")
pixels = image.reshape(-1, 3).astype(float) / 255.0
print("Разрешение:", image.shape[:2], "пикселей:", len(pixels))

with threadpool_limits(limits=1):
    initial_colors, _ = kmeans_plusplus(
        pixels, n_clusters=16, random_state=SEED
    )


def make_color_model(method):
    common = dict(
        n_clusters=16,
        init=initial_colors.copy(),
        n_init=1,
        random_state=SEED,
    )
    if method == "KMeans":
        return KMeans(**common, max_iter=300, tol=1e-4)
    return MiniBatchKMeans(
        **common,
        batch_size=1024,
        max_iter=100,
        tol=0.0,
        max_no_improvement=10,
    )


methods = ["KMeans", "MiniBatchKMeans"]
benchmark_rows = []
representatives = {}
order_rng = np.random.default_rng(SEED)
with threadpool_limits(limits=1):
    # Прогрев: полное обучение и предсказание, время не записываем.
    for method in methods:
        warmup = make_color_model(method).fit(pixels)
        warmup.predict(pixels)

    for repeat in range(5):
        for method in order_rng.permutation(methods):
            model = make_color_model(method)
            start = perf_counter()
            model.fit(pixels)
            fit_seconds = perf_counter() - start
            start = perf_counter()
            assignments = model.predict(pixels)
            predict_seconds = perf_counter() - start
            restored = model.cluster_centers_[assignments]
            benchmark_rows.append(
                {
                    "method": method,
                    "repeat": repeat + 1,
                    "fit_s": fit_seconds,
                    "predict_s": predict_seconds,
                    "MSE": np.mean((pixels - restored) ** 2),
                }
            )
            if repeat == 0:
                representatives[method] = (
                    model.cluster_centers_.copy(),
                    assignments.copy(),
                )

benchmark = pd.DataFrame(benchmark_rows)
timing_summary = (
    benchmark.groupby("method")
    .agg(
        fit_median_s=("fit_s", "median"),
        fit_min_s=("fit_s", "min"),
        fit_max_s=("fit_s", "max"),
        predict_median_s=("predict_s", "median"),
        predict_min_s=("predict_s", "min"),
        predict_max_s=("predict_s", "max"),
        MSE=("MSE", "mean"),
    )
    .reindex(methods)
)

speedup = (
    timing_summary.loc["KMeans", "fit_median_s"]
    / timing_summary.loc["MiniBatchKMeans", "fit_median_s"]
)
print(f"Отношение медиан fit KMeans / MiniBatchKMeans: {speedup:.2f}")



palette_16, full_indices = representatives["KMeans"]
indices_16 = full_indices.reshape(image.shape[:2]).astype(np.uint8)
reconstructed = {16: palette_16[full_indices].reshape(image.shape)}
mini_palette, mini_indices = representatives["MiniBatchKMeans"]
mini_restored = mini_palette[mini_indices]
color_results = []
with threadpool_limits(limits=1):
    for count in [8, 16, 64]:
        if count != 16:
            # Прогрев для данной конфигурации; n_init=3 как раньше.
            KMeans(n_clusters=count, n_init=3, random_state=SEED).fit(pixels)
            fit_times = []
            for repeat in range(5):
                model = KMeans(n_clusters=count, n_init=3, random_state=SEED)
                start = perf_counter()
                model.fit(pixels)
                fit_times.append(perf_counter() - start)
                if repeat == 0:
                    assignments = model.predict(pixels)
                    reconstructed[count] = model.cluster_centers_[
                        assignments
                    ].reshape(image.shape)
            fit_mean = np.mean(fit_times)
        else:
            fit_mean = benchmark.loc[
                benchmark.method == "KMeans", "fit_s"
            ].mean()
        restored = reconstructed[count].reshape(-1, 3)
        color_results.append(
            {
                "method": "KMeans",
                "colors": count,
                "MSE": np.mean((pixels - restored) ** 2),
                "fit_mean_s": fit_mean,
            }
        )
color_results.append(
    {
        "method": "MiniBatchKMeans",
        "colors": 16,
        "MSE": np.mean((pixels - mini_restored) ** 2),
        "fit_mean_s": benchmark.loc[
            benchmark.method == "MiniBatchKMeans", "fit_s"
        ].mean(),
    }
)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(image)
axes[0].set_title("Исходное: 427 × 640")
for count, ax in zip([8, 16, 64], axes[1:]):
    ax.imshow(np.clip(reconstructed[count], 0, 1))
    ax.set_title(f"{count} цветов")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(np.clip(reconstructed[16], 0, 1))
axes[0].set_title("KMeans: 16 цветов")
axes[1].imshow(np.clip(mini_restored.reshape(image.shape), 0, 1))
axes[1].set_title("MiniBatchKMeans: 16 цветов")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(pd.DataFrame(color_results).round(6).to_string(index=False))

### Как сохранить изображение компактнее: PNG с палитрой
Обычное RGB-изображение хранит три числа на пиксель — по одному для R, G и B.
После кластеризации достаточно сохранить:
1. **Палитру:** 16 RGB-цветов — округлённые центры кластеров.
2. **Индексы:** для каждого пикселя номер цвета от 0 до 15.

Например, вместо повторения `[120, 80, 40]` для множества пикселей сохраняем
этот цвет один раз под номером 3, а в изображении записываем `3, 3, 3, …`.
До PNG-кодирования RGB требует 24 бита на пиксель, а индекс из 16 вариантов —
4 бита, плюс небольшая палитра.

В Pillow режим `P` означает изображение с палитрой; `bits=4` задаёт 4-битные индексы PNG.
Сравним три файла **одного разрешения**:
- исходные цвета, RGB PNG;
- 16 цветов, RGB PNG;
- те же 16 цветов, PNG с палитрой.

Последние два файла декодируются в одинаковые RGB-пиксели. Их сравнение показывает
эффект способа хранения.

Нужен Pillow (`%pip install pillow`, если он не установлен).
Ячейка создаёт три файла в папке `clustering_color_compression` рядом с notebook.

In [ ]:
from pathlib import Path
from PIL import Image

compression_dir = Path("clustering_color_compression")
compression_dir.mkdir(exist_ok=True)
palette_rgb = np.rint(np.clip(palette_16, 0, 1) * 255).astype(np.uint8)
restored_rgb = palette_rgb[indices_16]

original_png = Image.fromarray(image)
rgb_png = Image.fromarray(restored_rgb)
indexed_png = Image.fromarray(indices_16).convert("P")
indexed_png.putpalette(palette_rgb.ravel().tolist())

files = [
    ("Исходное RGB", "original_rgb.png", original_png, {}),
    ("16 цветов, RGB", "16_colors_rgb.png", rgb_png, {}),
    ("16 цветов, палитра", "16_colors_palette.png", indexed_png, {"bits": 4}),
]
file_rows = []
for name, filename, png_image, options in files:
    path = compression_dir / filename
    png_image.save(path, format="PNG", optimize=True, **options)
    file_rows.append({"Вариант": name, "Байт": path.stat().st_size})

# Проверяем, что палитровый PNG восстанавливает ровно нужные цвета.
with Image.open(compression_dir / "16_colors_palette.png") as saved:
    assert np.array_equal(np.asarray(saved.convert("RGB")), restored_rgb)

sizes = pd.DataFrame(file_rows)
sizes["KiB"] = sizes["Байт"] / 1024
print(sizes.round(2).to_string(index=False))
ratio = sizes.iloc[0]["Байт"] / sizes.iloc[2]["Байт"]
print(f"Исходный RGB PNG / палитровый PNG: {ratio:.2f} раза")

Все три файла имеют одинаковое разрешение **427 × 640** и сохранены в PNG.

- *16 цветов, RGB* — Каждый пиксель заменён одним из 16 цветов, но хранится как RGB-тройка; размер пикселя тот же, но одинаковые цвета чаще повторяются, поэтому PNG сжимает лучше
- *16 цветов, палитра* — для каждого пикселя — только индекс 0–15; палитра из 16 RGB-цветов хранится отдельно; индексу достаточно 4 бит вместо 24 бит RGB

### Материалы
- [Лекция 3](https://github.com/gvolsky/hse-ml-course/tree/main/lecture3).
- [sklearn: кластеризация и метрики](https://scikit-learn.org/stable/modules/clustering.html).
- [SciPy linkage: определения расстояний и стоимость реализаций](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html).
- [DBSCAN: параметры](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html).